In [1]:
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pandas as pd



In [2]:
xgdf_train = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/train_df_20241211.parquet', engine='pyarrow')
xgdf_test = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/test_df_20241211.parquet', engine='pyarrow')
xgdf_val = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/val_df_20241211.parquet', engine='pyarrow')

xgdf = pd.concat([xgdf_train, xgdf_test, xgdf_val], axis=0, ignore_index=True)

In [ ]:
xgdf_train.head()

In [ ]:
# Define the specific item and store numbers you're interested in
# item_number = 105857
# store_number = 1

# Filter the dataset
# filtered_df = xgdf[(xgdf['store_nbr'] == store_number)]
filtered_df = xgdf
# Show the filtered dataset
print(filtered_df)


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Assuming you have the dataset in `df`
# Set 'date' as index
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
df = filtered_df.set_index('date')

# Predefined parameters
alpha = 0.5
beta = 0.3
gamma = 0.2
seasonal_periods = 52  # Assuming weekly seasonality
split_week = 189  # Cumulative week number for train-test split

# Initialize list to store RMSE results and predictions
rmse_results = []
all_predictions = []

# Group by store and item
for (store_nbr, item_nbr), group in df.groupby(['store_nbr', 'item_nbr']):
    # Split the data based on cumulative week number (<= 189 for train, > 189 for test)
    train = group[group['week_number_cum'] <= split_week]  # Train set: weeks <= 189
    test = group[(group['week_number_cum'] > split_week) & (group['week_number_cum'] <= 215)]
    # Test set: weeks > 189

    # Initialize storage for t+2 predictions
    t_plus_2_predictions = []
    test_dates = test.index

    # Rolling training data
    rolling_train = train.copy()

    # Iteratively forecast t+2
    for i in range(len(test)):
        # Fit the model on the rolling training data
        model = ExponentialSmoothing(
            rolling_train['unit_sales'],
            seasonal_periods=seasonal_periods,
            trend='add',
            seasonal='add'
        )
        fitted_model = model.fit(
            smoothing_level=alpha,
            smoothing_slope=beta,
            smoothing_seasonal=gamma,
            optimized=False
        )

        # Forecast t+2 (second step ahead)
        forecast = fitted_model.forecast(2)[-1]  # Get the second step forecast
        t_plus_2_predictions.append(forecast)

        # Add the observed test value to the rolling training data using pd.concat()
        rolling_train = pd.concat([rolling_train, test.iloc[[i]]])

    # Combine the test dates with the predictions and actual sales into a DataFrame
    predictions_df = pd.DataFrame({
        'store_nbr': store_nbr,
        'item_nbr': item_nbr,
        'date': test_dates,
        'actual_sales': test['unit_sales'].values,
        't+2_prediction': t_plus_2_predictions
    })

    # Append the predictions for this item-store combination to the list
    all_predictions.append(predictions_df)

    # Calculate RMSE for this item-store combination
    rmse = np.sqrt(((predictions_df['actual_sales'] - predictions_df['t+2_prediction']) ** 2).mean())
    rmse_results.append({
        'store_nbr': store_nbr,
        'item_nbr': item_nbr,
        'RMSE': rmse
    })

# Combine all predictions into a single DataFrame
final_predictions_df = pd.concat(all_predictions, ignore_index=True)

# Convert RMSE results to a DataFrame for easier inspection
rmse_df = pd.DataFrame(rmse_results)

# Display the RMSE results for each item-store combination
print(rmse_df)

# Display the final predictions DataFrame (including predictions for each item-store combination)
print(final_predictions_df)


In [ ]:
# Calculate the average RMSE across all item-store combinations
average_rmse = np.mean([result['RMSE'] for result in rmse_results])
# Display the average RMSE
print("\nAverage RMSE across all item-store combinations:", average_rmse)